# ELMo — Deep contextualized word representations (2018)

_Papers · Sequence & NLP_

**Read a sentence with a bidirectional LSTM language model, then take a learned weighted sum of its layers as the word's vector — so the same word gets a DIFFERENT vector in every context.**

---

This notebook is a practice scaffold for the **ELMo — Deep contextualized word representations (2018)** lesson. We build it up one step at a time — verify the layer-combination formula by hand, train a tiny bidirectional language model, then watch the word "bank" split into two senses (and collapse back when we ablate to a static embedding). Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q river
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Verify the ELMo combination formula (Equation 1)

ELMo turns a word's per-layer representations into one vector with `ELMo = gamma * sum_j s_j * h_j`, where the `s_j` are softmax-normalized layer weights and `gamma` is a scalar. Before training anything, we plug the lesson's worked numbers into this formula to confirm we reproduce its values to the digit.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# Verify the worked example (Equation 1) to the digit.
h0 = torch.tensor([1.0, 0.0, -1.0])
h1 = torch.tensor([0.0, 2.0, 0.0])
h2 = torch.tensor([1.0, 1.0, 1.0])
w = torch.tensor([0.0, 1.0, 2.0])
gamma = 2.0

s = F.softmax(w, dim=0)                          # softmax-normalized layer weights s_j
elmo = gamma * (s[0] * h0 + s[1] * h1 + s[2] * h2)   # Eq.(1): gamma * sum_j s_j h_j

print("s =", [round(x, 5) for x in s.tolist()])     # -> [0.09003, 0.24473, 0.66524]
print("ELMo (worked example) =", [round(x, 5) for x in elmo.tolist()])
#   -> [1.51054, 2.3094, 1.15042]  (matches the lesson's example, step by step)

### Step 2 — Build a tiny corpus and a per-layer biLM

To get *contextual* vectors we need a language model. We make a small corpus where "bank" means a riverbank in some sentences and a savings bank in others. The `DirLM` is a one-directional LSTM language model with one LSTM **per layer**, so we can expose each layer's hidden states — the `h_{k,j}` that ELMo mixes. We will run it forward and (on reversed text) backward to get a bidirectional model.

In [ ]:
# A tiny corpus where "bank" has two senses.
sents = ["the river bank was muddy", "we sat by the river bank",
         "the savings bank was open", "she went to the savings bank",
         "the boat reached the river bank", "he opened a savings bank account"]
words = sorted(set(" ".join(sents).split()))
stoi = {w: i for i, w in enumerate(["<bos>", "<eos>"] + words)}
itos = {i: w for w, i in stoi.items()}
V = len(stoi)

def enc(s):
    return [stoi["<bos>"]] + [stoi[w] for w in s.split()] + [stoi["<eos>"]]

data = [torch.tensor(enc(s)) for s in sents]

# The biLM: a forward LM and a backward LM (each L=2 LSTM layers).
D, H, L = 16, 16, 2

class DirLM(nn.Module):                          # one-directional LSTM language model
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, D)
        # one LSTM PER layer so we can expose each layer's output (h_{k,1}, h_{k,2}, ...)
        self.cells = nn.ModuleList([
            nn.LSTM(D if i == 0 else H, H, batch_first=True) for i in range(L)
        ])
        self.head = nn.Linear(H, V)
    def layers(self, ids):                       # return [h0, h1, ..., hL], each [T, .]
        reps = [self.emb(ids)]                    # h_{k,0} = static, context-INDEPENDENT token vector
        cur = self.emb(ids).unsqueeze(0)          # [1, T, D]
        for cell in self.cells:                   # run layer by layer to expose each layer's states
            cur, _ = cell(cur)
            reps.append(cur.squeeze(0))           # h_{k,j} for j = 1..L
        return reps
    def forward(self, ids):
        return self.head(self.layers(ids)[-1])    # predict next token from the TOP layer

### Step 3 — Train the forward and backward language models

We train two `DirLM`s jointly: the forward LM predicts each next token from the prefix, and the backward LM does the same on the reversed sentence. Their parameters share one optimizer. After a few hundred epochs the loss settles low — the model has learned the toy grammar well enough to give context-sensitive hidden states.

In [ ]:
fwd, bwd = DirLM(), DirLM()
opt = torch.optim.Adam(list(fwd.parameters()) + list(bwd.parameters()), lr=0.02)

for epoch in range(400):
    opt.zero_grad()
    loss = 0.0
    for ids in data:
        # forward LM: predict t_{k+1} from t_1..t_k
        logit_f = fwd(ids[:-1])
        loss += F.cross_entropy(logit_f, ids[1:])
        # backward LM: same on the reversed sentence
        rev = torch.flip(ids, [0])
        logit_b = bwd(rev[:-1])
        loss += F.cross_entropy(logit_b, rev[1:])
    loss.backward()
    opt.step()

print("biLM final loss =", round(float(loss), 4))

### Step 4 — Combine layers into ELMo vectors and compare senses

Now we apply Equation 1 for real. `elmo_vec` aligns the forward and backward per-layer reps for a target word, concatenates them, then takes the weighted sum. With weight on the **contextual** LSTM layers the two "bank" vectors should point in different directions (cosine well below 1). As an ablation, putting all weight on **layer 0** (the static, context-free embedding) should fuse them back to nearly the same vector.

In [ ]:
# ELMo combine: gamma * sum_j s_j h_j over the 2L+1 representations.
@torch.no_grad()
def elmo_vec(sentence, target, s_weights, gamma):
    ids = torch.tensor(enc(sentence))
    toks = ["<bos>"] + sentence.split() + ["<eos>"]
    k = toks.index(target)
    fr = fwd.layers(ids)                          # forward per-layer reps  [h0, h1, h2]
    # backward reps, re-aligned to the original left-to-right order
    br = [torch.flip(r, [0]) for r in bwd.layers(torch.flip(ids, [0]))]
    layers = [torch.cat([fr[j][k], br[j][k]]) for j in range(L + 1)]   # h_{k,j} = [fwd; bwd]
    s = F.softmax(torch.tensor(s_weights), dim=0)
    return gamma * sum(s[j] * layers[j] for j in range(L + 1))

def cos(a, b):
    return float(F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)))

# CONTEXTUAL: weight the LSTM layers -> "bank" should differ across the two senses
sw = [0.0, 1.0, 1.0]
g = 1.0
v_river = elmo_vec("the river bank was muddy", "bank", sw, g)
v_money = elmo_vec("the savings bank was open", "bank", sw, g)
print("contextual cos(river bank, savings bank) =", round(cos(v_river, v_money), 4))

# ABLATION: put ALL weight on layer 0 (context-independent) -> static embedding
sw_static = [10.0, 0.0, 0.0]
v_river_s = elmo_vec("the river bank was muddy", "bank", sw_static, g)
v_money_s = elmo_vec("the savings bank was open", "bank", sw_static, g)
print("static(layer-0) cos(river bank, savings bank) =", round(cos(v_river_s, v_money_s), 4))
# Expect: contextual cosine clearly < 1 (two senses separate); static cosine ~ 1.0 (senses fuse).

## Visualize the data & results

_Does the SAME word 'bank' get two different vectors in 'river bank' vs 'savings bank' under contextual ELMo, and do those two vectors collapse back into one when we ablate to a static (layer-0-only) embedding?_

### Step 1 — Recompute the two "bank" vectors each way

This reuses the biLM trained above. We rebuild the "bank" vector in both sentences twice: once trusting the contextual LSTM layers (`[0,1,1]`), once ablated to the static layer-0 embedding (`[10,0,0]`). That gives us four vectors — the river/savings pair under each setting — ready to compare.

In [ ]:
# (Reuses the biLM trained in the CODE cell above.)
import torch.nn.functional as F

def cos(a, b):
    return round(float(F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0))), 4)

g = 1.0
# contextual: trust the LSTM layers
c_river = elmo_vec("the river bank was muddy", "bank", [0.0, 1.0, 1.0], g)
c_money = elmo_vec("the savings bank was open", "bank", [0.0, 1.0, 1.0], g)
# static ablation: all weight on layer 0
s_river = elmo_vec("the river bank was muddy", "bank", [10.0, 0.0, 0.0], g)
s_money = elmo_vec("the savings bank was open", "bank", [10.0, 0.0, 0.0], g)

### Step 2 — Compare the cosines: do the senses separate?

Now we measure the cosine similarity between the river and savings "bank" vectors under each setting. Under the contextual layers the cosine should be well below 1 (the two senses point in different directions); under the static ablation it should sit near 1.0 (the senses collapse into one vector) — the exact static-embedding limitation ELMo fixes.

In [ ]:
print("contextual cosine:", cos(c_river, c_money))   # ~0.32  -> two senses separate
print("static     cosine:", cos(s_river, s_money))   # ~1.00  -> two senses fuse
# Bar chart: [contextual ~0.32, static ~1.00].

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** With L=1 biLM, layer vectors $h_0=(2,0)$ and $h_1=(0,2)$, unnormalized weights $w=(0,0)$ (equal),
            and $\gamma=1$, compute $\text{ELMo}$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Softmax equal weights (0,0): $s_0=s_1=0.5$. — _Equal logits give equal softmax probabilities._
- Weighted sum: $0.5(2,0)+0.5(0,2)=(1,1)$. — _Each layer contributes half._
- Scale by $\gamma=1$: unchanged. — _Multiplying by 1 does nothing._

**Answer:** $\text{ELMo}=(1,1)$ — a 50/50 blend of the static layer and the LSTM layer.

</details>

**Problem 2.** Same as above but the task learns to trust the contextual layer: unnormalized weights $w=(0,3)$,
            $\gamma=1$. Does the result move toward the static layer or the contextual layer?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Softmax (0,3): $e^0=1$, $e^3=20.0855$, sum $21.0855$, so $s_0=0.0474$, $s_1=0.9526$. — _The larger logit (3) gets almost all the weight._
- Weighted sum: $0.0474(2,0)+0.9526(0,2)=(0.0948,\ 1.9052)$. — _The contextual layer $h_1=(0,2)$ now dominates._

**Answer:** $\text{ELMo}\approx(0.095,\ 1.905)$ — pulled almost entirely onto the contextual layer, away from the static one.

</details>

**Problem 3.** ABLATION. Force ELMo to be a static embedding by putting ALL the weight on layer 0:
            unnormalized weights $w=(10,0,0)$ for the L=2 example with $h_0=(1,0,-1)$, $h_1=(0,2,0)$,
            $h_2=(1,1,1)$, $\gamma=1$. What is ELMo, and what does this tell you about the two "bank" senses?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Softmax (10,0,0): $s_0\approx0.9999$, $s_1,s_2\approx0$. — _One huge logit swamps the softmax._
- Weighted sum $\approx h_0=(1,0,-1)$. — _Only layer 0 survives — and layer 0 is context-independent._
- Conclude both sentences give the SAME vector for "bank". — _Layer 0 is identical regardless of context, so the two senses fuse._

**Answer:** $\text{ELMo}\approx(1,0,-1)$ regardless of sentence. Collapsing to layer 0 throws away context — exactly the static-embedding limitation ELMo was built to fix. This is the CODEVIZ ablation: with the contextual layers removed, the two "bank" vectors become identical.

</details>